# BERT Fine-tuning for Author Profiling

Fine-tunes `bert-base-uncased` on `pan14_prepared_with_reviews.csv` for age or gender prediction.

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4)
2. Upload `pan14_prepared_with_reviews.csv` to your Google Drive under `My Drive/thesis/`
3. Set `TASK = "age"` or `TASK = "gender"` in the Config cell
4. Run all cells top to bottom

Run twice (once per task). Model saves to `My Drive/thesis/bert_age_model/` or `bert_gender_model/`.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Check GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU found — go to Runtime > Change runtime type > GPU')

In [ ]:
# Upgrade transformers and accelerate to latest compatible versions
!pip install -q -U transformers accelerate
import transformers
print(f'transformers: {transformers.__version__}')

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────
TASK        = "age"       # change to "gender" for gender task
MODEL_NAME  = "bert-base-uncased"

CSV_FILE    = "/content/drive/MyDrive/thesis/pan14_colab.csv"   # pre-cleaned, 20MB
OUTPUT_DIR  = f"/content/drive/MyDrive/thesis/bert_{TASK}_model"

MAX_LENGTH        = 64
TRAIN_BATCH_SIZE  = 8
EVAL_BATCH_SIZE   = 16
GRAD_ACCUM_STEPS  = 2     # effective batch = 16
NUM_EPOCHS        = 3
LEARNING_RATE     = 2e-5
WEIGHT_DECAY      = 0.01
RANDOM_STATE      = 42
# ──────────────────────────────────────────────────────────────────────────
print(f'Task: {TASK} | Output: {OUTPUT_DIR}')

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        weights = self.class_weights.to(outputs.logits.device)
        loss = nn.CrossEntropyLoss(weight=weights)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            texts, truncation=True, padding='max_length',
            max_length=max_length, return_tensors=None
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1,
        'macro_precision': precision,
        'macro_recall': recall,
    }

print('Imports OK')

In [ ]:
# Load and inspect data  (pan14_colab.csv is pre-cleaned — C engine works fine)
import gc

df = pd.read_csv(CSV_FILE)   # fast C engine, no parsing issues
df = df.dropna(subset=['text', TASK]).copy()
df['text'] = df['text'].astype(str)
gc.collect()
print(f'Rows: {len(df)} | Avg text len: {df["text"].str.len().mean():.0f} chars')

labels_sorted = sorted(df[TASK].unique().tolist())
label2id = {label: idx for idx, label in enumerate(labels_sorted)}
id2label  = {idx: label for label, idx in label2id.items()}
df['label'] = df[TASK].map(label2id)

print(f'Labels: {label2id}')
print('Class distribution:')
print(df[TASK].value_counts())

In [ ]:
# Class weights + train/val split
weights_array = compute_class_weight('balanced', classes=np.array(labels_sorted), y=df[TASK].tolist())
class_weights = torch.tensor(weights_array, dtype=torch.float)
print(f'Class weights: { {labels_sorted[i]: round(class_weights[i].item(), 3) for i in range(len(labels_sorted))} }')

train_df, val_df = train_test_split(df, test_size=0.1, random_state=RANDOM_STATE, stratify=df['label'])
print(f'Train: {len(train_df)} | Val: {len(val_df)}')

os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(os.path.join(OUTPUT_DIR, 'label_mapping.json'), 'w') as f:
    json.dump({'task': TASK, 'label2id': label2id, 'id2label': {str(k): v for k, v in id2label.items()}}, f, indent=2)
print('Label mapping saved')

In [ ]:
# Tokenize
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizing training set...')
train_dataset = TextDataset(train_df['text'].tolist(), train_df['label'].tolist(), tokenizer, MAX_LENGTH)
print('Tokenizing validation set...')
val_dataset   = TextDataset(val_df['text'].tolist(),   val_df['label'].tolist(),   tokenizer, MAX_LENGTH)
print(f'Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}')

In [ ]:
# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(labels_sorted), id2label=id2label, label2id=label2id
)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Train
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,      # trades compute for VRAM — critical on free T4
    dataloader_pin_memory=False,      # reduces CPU RAM pressure
    report_to='none',
    seed=RANDOM_STATE,
)

trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

start = time.time()
trainer.train()
print(f'Training done in {round((time.time()-start)/60, 1)} minutes')

In [ ]:
# Save model to Drive
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to: {OUTPUT_DIR}')

In [ ]:
# Final evaluation on validation set
metrics = trainer.evaluate()
print('\nValidation metrics:')
for k, v in metrics.items():
    print(f'  {k}: {round(v, 4) if isinstance(v, float) else v}')

preds = trainer.predict(val_dataset)
pred_ids  = preds.predictions.argmax(axis=1)
true_ids  = preds.label_ids
print('\nClassification report:')
print(classification_report(true_ids, pred_ids, target_names=labels_sorted, zero_division=0))

## Evaluate on LiLaH (cross-dataset)

Upload `hate_speech_only.tsv` to `My Drive/thesis/` first, then run the two cells below.

In [ ]:
# ── LiLaH evaluation config ───────────────────────────────────────────────
LILAH_FILE       = "/content/drive/MyDrive/thesis/hate_speech_only.tsv"
LILAH_OUTPUT_CSV = f"/content/drive/MyDrive/thesis/bert_{TASK}_lilah_predictions.csv"

# Use the best complete checkpoint (the one with model.safetensors confirmed present).
# checkpoint-1340 = epoch 2 (macro F1 0.4118 for age; adjust path for gender).
EVAL_MODEL_DIR = OUTPUT_DIR + "/checkpoint-1340"
# ──────────────────────────────────────────────────────────────────────────
print(f'Evaluating on LiLaH | Task: {TASK}')
print(f'Model: {EVAL_MODEL_DIR}')
print(f'LiLaH file: {LILAH_FILE}')

In [ ]:
# Run LiLaH evaluation
import json
from sklearn.metrics import classification_report, confusion_matrix

# Label mapping (saved in root OUTPUT_DIR)
with open(os.path.join(OUTPUT_DIR, 'label_mapping.json')) as f:
    mapping = json.load(f)
label2id_eval = mapping['label2id']
id2label_eval = {int(k): v for k, v in mapping['id2label'].items()}
print(f'Labels: {label2id_eval}')

# Load model from checkpoint; tokenizer from HuggingFace (unmodified for BERT)
eval_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
eval_model     = AutoModelForSequenceClassification.from_pretrained(EVAL_MODEL_DIR)
eval_model.eval()
eval_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
eval_model.to(eval_device)
print('Model loaded')

# Load LiLaH
lilah_df = pd.read_csv(LILAH_FILE, sep='\t')
lilah_df = lilah_df.dropna(subset=['text', TASK]).copy()
lilah_df['text'] = lilah_df['text'].astype(str)

# Normalise labels to match training label space
AGE_MAP    = {'0-25':'0-25','26-35':'26-35','36-65':'36-65','66-':'66-',
              '18-24':'0-25','25-34':'26-35','35-49':'36-65','50-64':'36-65','65-xx':'66-','65+':'66-'}
GENDER_MAP = {'male':'M','female':'F','m':'M','f':'F','M':'M','F':'F'}
norm_map   = AGE_MAP if TASK == 'age' else GENDER_MAP
lilah_df[TASK] = lilah_df[TASK].astype(str).str.strip().map(lambda x: norm_map.get(x, x))
lilah_df = lilah_df[lilah_df[TASK].isin(label2id_eval)].copy()
print(f'LiLaH rows: {len(lilah_df)}')
print(lilah_df[TASK].value_counts())

# Predict in batches to avoid OOM
BATCH = 64
all_preds = []
texts_all = lilah_df['text'].tolist()
for i in range(0, len(texts_all), BATCH):
    batch_texts = texts_all[i:i+BATCH]
    enc = eval_tokenizer(batch_texts, truncation=True, padding=True,
                         max_length=MAX_LENGTH, return_tensors='pt')
    enc = {k: v.to(eval_device) for k, v in enc.items()}
    with torch.no_grad():
        logits = eval_model(**enc).logits
    all_preds.extend(torch.argmax(logits, dim=1).cpu().tolist())

pred_labels = [id2label_eval[i] for i in all_preds]
true_labels = lilah_df[TASK].tolist()

print('\n=== LiLaH Cross-Dataset Results ===')
print(classification_report(true_labels, pred_labels, zero_division=0))

# Save predictions to Drive
out_df = lilah_df.copy()
out_df['predicted'] = pred_labels
out_df.to_csv(LILAH_OUTPUT_CSV, index=False)
print(f'Predictions saved to: {LILAH_OUTPUT_CSV}')